In [88]:
import sys
sys.path.append('..')

from collections import defaultdict
from adaptation.misc import NameAnonymizer
from datetime import datetime
import pandas as pd
import json
import os


In [89]:
data_dir = "./data"
raw_dir = os.path.join(data_dir, "raw")
pretest_dir = os.path.join(raw_dir, "pretest")
postest_dir = os.path.join(raw_dir, "postest")
gameplay_raw_path = os.path.join(raw_dir, "gameplay", "full_xapi_data_33.ndjson")

processed_dir = os.path.join(data_dir, "processed")
pretest_path = os.path.join(processed_dir, "pretest.csv")
postest_path = os.path.join(processed_dir, "postest.csv")
gameplay_path = os.path.join(processed_dir, "gameplay.ndjson")

adaptation_dir = os.path.join("../adaptation", "data")
metadata_path = os.path.join(adaptation_dir, "metadata.json")
situation_dir = os.path.join(adaptation_dir, "processed")
whitelist_path = os.path.join(adaptation_dir, "name_whitelist.txt")
names_path = os.path.join(adaptation_dir, "nombres-propios-es.txt")


In [90]:
with open(metadata_path, "r", encoding="utf-8") as f:
	config = json.load(f)

node_order = []
node_context = {}

for key in sorted(config.keys(), key=int):
	item = config[key]

	scene = os.path.splitext(os.path.basename(item["file"]))[0]
	scene = scene[0].upper() + scene[1:]

	part = item["object"].split(".")[0]

	node = f"{scene}.{part}"

	node_order.append(node)

print(node_order)


['Scene1Classroom.part2', 'Scene1Bedroom1.computer1', 'Scene1Bedroom1.computer2', 'Scene1Bedroom2.computer', 'Scene2Break.part2', 'Scene3Bedroom.main', 'Scene4Backyard.mainConversation', 'Scene4Garage.phone1', 'Scene4Garage.photo', 'Scene4Bedroom.phone', 'Scene6BedroomRouteA1.phone', 'Scene6LunchRouteB.phone']


In [91]:
pretest_df = pd.read_csv(pretest_path, encoding="utf-8")
postest_df = pd.read_csv(postest_path, encoding="utf-8")


In [92]:
def extract_likert(value: str):
	return int(value.split(" - ")[0])


In [93]:
def load_json_files(dir: str) -> list[dict]:
	data = []

	for filename in os.listdir(dir):
		if filename.endswith(".json"):
			filepath = os.path.join(dir, filename)

			with open(filepath, "r", encoding="utf-8") as f:
				data.append(json.load(f))

	return data

In [94]:
pretests = load_json_files(pretest_dir)
	

In [95]:
rows = []

for pretest in pretests:
	# Las cuatro últimas respuestas corresponden a las preguntas likert
	values = list(pretest.values())
	likert_values = values[-4:]

	row = {
		"access_code": pretest["Código de acceso"],
		"age": int(pretest["Edad"]),
		"gender": pretest["Género"] or pretest["Género [Otro]"],
		"education_level": pretest["Nivel de estudios actual"],
		"study_area": pretest["Área principal de estudios"] or pretest["Área principal de estudios [Otro]"],
		"dialogue_advance_only": extract_likert(likert_values[0]),
		"dialogue_predefined_options": extract_likert(likert_values[1]),
		"dialogue_free_text": extract_likert(likert_values[2]),
		"dialogue_hybrid": extract_likert(likert_values[3]),
	}

	rows.append(row)

df = pd.DataFrame(rows)


In [96]:
print(df.dtypes)


access_code                      str
age                            int64
gender                           str
education_level                  str
study_area                       str
dialogue_advance_only          int64
dialogue_predefined_options    int64
dialogue_free_text             int64
dialogue_hybrid                int64
dtype: object


In [97]:
display(df)


,access_code,age,gender,education_level,study_area,dialogue_advance_only,dialogue_predefined_options,dialogue_free_text,dialogue_hybrid
0,22_gmvf,22,Masculino,Estudios universitarios,Ciencias de la Salud,2,4,3,4
1,22_tstp,22,Femenino,Estudios universitarios,Ciencias de la Salud,2,4,5,4
2,22_vezs,24,Femenino,Estudios universitarios,Ciencias Sociales,3,3,3,3
3,22_wuak,65,Femenino,Estudios universitarios,Educación,5,1,1,1


In [98]:
df.to_csv(pretest_path, index=False, encoding="utf-8")


In [99]:
postests = load_json_files(postest_dir)
			

In [100]:
rows = []

for postest in postests:
	values = list(postest.values())
	
	# Saltarse metadata
	# response_id, submission_date, last_page, language, seed,
	# access_code, start_date, last_action_date
	likert_values = values[8:-2]

	# Dividir en secciones
	sus = likert_values[:10]
	free_text = likert_values[10:13]
	hybrid = likert_values[13:17]
	preferences = likert_values[17:21]

	row = {
		"access_code": postest["Código de acceso"],

		# SUS
		"sus_use_frequently": extract_likert(sus[0]),
		"sus_unnecessarily_complex": extract_likert(sus[1]),
		"sus_easy_to_use": extract_likert(sus[2]),
		"sus_need_technical_support": extract_likert(sus[3]),
		"sus_well_integrated": extract_likert(sus[4]),
		"sus_too_much_inconsistency": extract_likert(sus[5]),
		"sus_easy_to_learn": extract_likert(sus[6]),
		"sus_cumbersome": extract_likert(sus[7]),
		"sus_confident": extract_likert(sus[8]),
		"sus_need_to_learn": extract_likert(sus[9]),

		# Texto libre
		"free_text_easy_to_express": extract_likert(free_text[0]),
		"free_text_understood": extract_likert(free_text[1]),
		"free_text_coherent_responses": extract_likert(free_text[2]),

		# Diálogos híbridos
		"hybrid_more_natural": extract_likert(hybrid[0]),
		"hybrid_more_freedom": extract_likert(hybrid[1]),
		"hybrid_improves_interaction": extract_likert(hybrid[2]),
		"hybrid_overall_satisfaction": extract_likert(hybrid[3]),

		# Preferencia de diálogos
		"dialogue_advance_only": extract_likert(preferences[0]),
		"dialogue_predefined_options": extract_likert(preferences[1]),
		"dialogue_free_text": extract_likert(preferences[2]),
		"dialogue_hybrid": extract_likert(preferences[3]),

		# Preguntas abiertas
		"liked_most": values[-2],
		"comments": values[-1],
	}

	rows.append(row)

df = pd.DataFrame(rows)


In [101]:
print(df.dtypes)


access_code                       str
sus_use_frequently              int64
sus_unnecessarily_complex       int64
sus_easy_to_use                 int64
sus_need_technical_support      int64
sus_well_integrated             int64
sus_too_much_inconsistency      int64
sus_easy_to_learn               int64
sus_cumbersome                  int64
sus_confident                   int64
sus_need_to_learn               int64
free_text_easy_to_express       int64
free_text_understood            int64
free_text_coherent_responses    int64
hybrid_more_natural             int64
hybrid_more_freedom             int64
hybrid_improves_interaction     int64
hybrid_overall_satisfaction     int64
dialogue_advance_only           int64
dialogue_predefined_options     int64
dialogue_free_text              int64
dialogue_hybrid                 int64
liked_most                        str
comments                          str
dtype: object


In [102]:
display(df)


,access_code,sus_use_frequently,sus_unnecessarily_complex,sus_easy_to_use,sus_need_technical_support,sus_well_integrated,sus_too_much_inconsistency,sus_easy_to_learn,sus_cumbersome,sus_confident,...,hybrid_more_natural,hybrid_more_freedom,hybrid_improves_interaction,hybrid_overall_satisfaction,dialogue_advance_only,dialogue_predefined_options,dialogue_free_text,dialogue_hybrid,liked_most,comments
0,22_gmvf,3,1,4,2,4,1,4,1,5,...,4,1,4,4,4,4,2,3,La habilidad de la IA generativa para desarrol...,"Deberían ser más permisivas las respuestas, ya..."
1,22_tstp,4,5,4,3,3,1,4,5,4,...,4,3,4,5,2,4,4,5,,Algunas partes exigia que respondieses una cos...
2,22_vezs,5,1,3,1,4,1,4,1,3,...,3,4,4,4,3,3,3,3,que sea automatizado,siento que esta bien equilibrado
3,22_wuak,4,3,4,2,4,2,5,2,4,...,4,4,5,5,5,4,3,3,,


In [103]:
df.to_csv(postest_path, index=False, encoding="utf-8")


In [104]:
with open(gameplay_raw_path, "r", encoding="utf-8") as f:
	records = [json.loads(line) for line in f]

print(f"Original records: {len(records)}")


Original records: 4925


In [105]:
# USUARIOS
users = {"vezs", "tstp", "wuak", "gmvf"}

records = [
	record for record in records
	if record["actor"]["account"]["name"] in users
]

print(f"Filtered records: {len(records)}")


Filtered records: 3839


In [106]:
THOUGHTBOX_EVENT_IDS = {
	"https://w3id.org/xapi/lab/activity-types/story-node/ThoughtBoxStart",
	"https://w3id.org/xapi/lab/activity-types/story-node/ThoughtBoxEnd",
}

DIALOG_TREE_PREFIX = "https://w3id.org/xapi/seriousgames/activity-types/dialog-tree/"

MATCHING_TEXT_EXTENSION = "https://w3id.org/xapi/seriousgame/extensions/MatchingText"
METHOD_EXTENSION = "https://w3id.org/xapi/seriousgame/extensions/Method"

LSTM_USERS = {"vezs", "tstp"}
# LSTM_USERS = {}

filtered_records = []

for record in records:
	object_id = record.get("object", {}).get("id", "")

	if object_id in THOUGHTBOX_EVENT_IDS:
		filtered_records.append(record)
		continue

	if object_id.startswith(DIALOG_TREE_PREFIX):
		extensions = record.get("result", {}).get("extensions", {})

		if MATCHING_TEXT_EXTENSION in extensions:
			user = record.get("actor", {}).get("account", {}).get("name")

			if user in LSTM_USERS:
				extensions[METHOD_EXTENSION] = "lstm"
			
			filtered_records.append(record)

records = filtered_records

print(f"Records after event filtering: {len(records)}")


Records after event filtering: 240


In [107]:
records.sort(key=lambda record: record["timestamp"])

print(f"Records after event filtering: {len(records)}")


Records after event filtering: 240


In [108]:
print(node_order)

['Scene1Classroom.part2', 'Scene1Bedroom1.computer1', 'Scene1Bedroom1.computer2', 'Scene1Bedroom2.computer', 'Scene2Break.part2', 'Scene3Bedroom.main', 'Scene4Backyard.mainConversation', 'Scene4Garage.phone1', 'Scene4Garage.photo', 'Scene4Bedroom.phone', 'Scene6BedroomRouteA1.phone', 'Scene6LunchRouteB.phone']


In [109]:
situation_files = []

for file in os.listdir(situation_dir):
	if file.startswith("situation_") and file.endswith(".csv"):
		situation_files.append(file)

situation_files.sort(
	key=lambda file: int(file.replace("situation_", "").replace(".csv", ""))
)

situation_dfs = {}

for file in situation_files:
	situation_number = int(file.replace("situation_", "").replace(".csv", ""))
	situation_dfs[situation_number] = pd.read_csv(os.path.join(situation_dir, file))

print(len(situation_dfs))


12


In [110]:
name_anonymizer = NameAnonymizer(
	names_path=names_path,
	whitelist_path=whitelist_path,
	replacement="{{name}}"
)


In [111]:
BRANCH_EXTENSION = "https://w3id.org/xapi/seriousgame/extensions/Branch"

for record in records:
	object_id = record.get("object", {}).get("id", "")

	if object_id.startswith(DIALOG_TREE_PREFIX):
		extensions = record.get("result", {}).get("extensions", {})

		# Solo procesar si el nodo no tiene Branch
		if BRANCH_EXTENSION not in extensions:
			# Quitar el prefijo del dialog tree
			node_name = object_id[len(DIALOG_TREE_PREFIX):]

			matching_text = extensions.get(MATCHING_TEXT_EXTENSION)

			# Quedarse con las dos primeras partes separadas por "."
			node_prefix = ".".join(node_name.split(".")[:2])
			node_prefix_parts = node_prefix.split(".")

			situation_number = None
			fallback_situation = None

			# print(node_order)
			for i, expected in enumerate(node_order, start=1):
				if expected == node_prefix:
					situation_number = i
					break

				expected_parts = expected.split(".")

				if fallback_situation is None:
					expected_parts = expected.split(".")
					if expected_parts[1] == node_prefix_parts[1]:
						fallback_situation = i

			if situation_number is None:
				situation_number = fallback_situation

			situation_df = situation_dfs[situation_number]

			matching_text_clean = name_anonymizer.anonymize_names(matching_text)
			match = situation_df[situation_df["text_clean"] == matching_text_clean]

			if not match.empty:
				choice = match.iloc[0]["choice"]
			else:
				raise ValueError(f"No se encontró ninguna choice para '{matching_text_clean}' (situación {situation_number}, nodo {node_prefix})")

			extensions[BRANCH_EXTENSION] = choice

			print(node_prefix, situation_number, matching_text, choice)
		

Scene1Classroom.part2 1 Holaaa buenas, soy cris, acabo de llegar y estoy todavía un poco perdido, y tú? friendly
Scene1Classroom.part2 1 Hola shy
Computer.computer1 2 Hey, si soy cris, me gusta mucho jajajja msg4
Computer.computer2 3 Puff desde hace mucho tiempo 2yearsResponse
Computer.computer 4 Ay pobre, dormiste mal o algo? ask2
Computer.computer 4 Ay pobre, dormiste mal o algo? ask2
Computer.computer 4 Ahh, me alegra que estés mejor ask2
Scene2Break.part2 5 Acepto aunque no sé si podré unsure
Scene3Bedroom.main 6 Bastante, no sé le da mal jajaja. Y a ti?? liked2
Scene3Bedroom.main 6 Bro siempre pienso que no puede superarse y siempre hace algo mejor liked2
Scene3Bedroom.main 6 Me ha encantado ha sido bastante divertido liked2
Scene4Backyard.mainConversation 7 Bien :) good2
Scene4Garage.phone1 8 Le han encantado!! Le han regalado ... msg2
Scene4Garage.photo 9 No truth2
Scene4Garage.photo 9 No truth2
Scene4Bedroom.phone 10 Ya acabo la fiesta, que tal tu noche? msg3
Scene4Bedroom.phon

In [112]:
NODE_EXTENSION = "https://w3id.org/xapi/seriousgame/extensions/Node"

for record in records:
	event = record.get("object", {}).get("id", "")

	if event in THOUGHTBOX_EVENT_IDS:
		extensions = record.get("result", {}).get("extensions", {})

		user = record.get("actor", {}).get("account", {}).get("name", "unknown")
		timestamp = record.get("timestamp", "unknown")
		node = extensions.get(NODE_EXTENSION, "unknown")

		event_name = event.split("/")[-1]

		print(f"{timestamp} | {user} | {event_name} | {node}")


2026-07-23T22:29:10.882Z | vezs | ThoughtBoxStart | Scene1Classroom.part2.thanks_similarity
2026-07-23T22:29:10.884Z | vezs | ThoughtBoxEnd | Scene1Classroom.part2.thanks_similarity
2026-07-23T22:29:10.891Z | vezs | ThoughtBoxStart | Scene1Classroom.part2.thanks_similarity
2026-07-23T22:29:10.893Z | vezs | ThoughtBoxEnd | Scene1Classroom.part2.thanks_similarity
2026-07-23T22:31:47.465Z | vezs | ThoughtBoxStart | Computer.computer1.choices_similarity
2026-07-23T22:31:47.467Z | vezs | ThoughtBoxEnd | Computer.computer1.choices_similarity
2026-07-23T22:31:47.507Z | vezs | ThoughtBoxStart | Computer.computer2.root
2026-07-23T22:31:47.508Z | vezs | ThoughtBoxEnd | Computer.computer2.root
2026-07-23T22:34:54.627Z | vezs | ThoughtBoxStart | Computer.computer.choices2_similarity
2026-07-23T22:34:54.628Z | vezs | ThoughtBoxEnd | Computer.computer.choices2_similarity
2026-07-23T22:34:54.631Z | vezs | ThoughtBoxStart | Computer.computer.choices2_similarity
2026-07-23T22:34:54.632Z | vezs | Though

In [113]:
for record in records:
	event = record.get("object", {}).get("id", "")

	if event.startswith(DIALOG_TREE_PREFIX):
		extensions = record.get("result", {}).get("extensions", {})
		
		user = record.get("actor", {}).get("account", {}).get("name", "unknown")
		timestamp = record.get("timestamp", "unknown")
		matching_text = extensions.get(MATCHING_TEXT_EXTENSION, "unknown")

		event_name = event.split("/")[-1]

		print(f"{timestamp} | {user} | {event_name} | {matching_text}")
	

2026-07-23T22:29:10.883Z | vezs | Scene1Classroom.part2.thanks_similarity | Holaaa buenas, soy cris, acabo de llegar y estoy todavía un poco perdido, y tú?
2026-07-23T22:29:10.892Z | vezs | Scene1Classroom.part2.thanks_similarity | Hola
2026-07-23T22:31:47.466Z | vezs | Computer.computer1.choices_similarity | Hey, si soy cris, me gusta mucho jajajja
2026-07-23T22:31:47.508Z | vezs | Computer.computer2.root | Puff desde hace mucho tiempo
2026-07-23T22:34:54.627Z | vezs | Computer.computer.choices2_similarity | Ay pobre, dormiste mal o algo?
2026-07-23T22:34:54.631Z | vezs | Computer.computer.choices2_similarity | Ay pobre, dormiste mal o algo?
2026-07-23T22:34:54.636Z | vezs | Computer.computer.choices2_similarity | Ahh, me alegra que estés mejor
2026-07-23T22:37:15.427Z | vezs | Scene2Break.part2.choice_similarity | Acepto aunque no sé si podré
2026-07-23T22:39:34.051Z | vezs | Scene3Bedroom.main.choices_similarity | Bastante, no sé le da mal jajaja. Y a ti??
2026-07-23T22:39:34.056Z |

In [117]:
START_EVENT = "https://w3id.org/xapi/lab/activity-types/story-node/ThoughtBoxStart"
END_EVENT = "https://w3id.org/xapi/lab/activity-types/story-node/ThoughtBoxEnd"

# Comienzos para cada (usuario, nodo)
pending_starts = defaultdict(list)

# Duraciones para cada (usuario, nodo)
durations = defaultdict(list)

matches = defaultdict(list)

for record in records:
	event = record.get("object", {}).get("id")
	user = record.get("actor", {}).get("account", {}).get("name")
	node = record.get("result", {}).get("extensions", {}).get(NODE_EXTENSION)

	timestamp = datetime.fromisoformat(record["timestamp"].replace("Z", "+00:00"))

	key = (user, node)

	if event == START_EVENT:
		pending_starts[key].append(timestamp)

	elif event == END_EVENT:
		if pending_starts[key]:
			start = pending_starts[key].pop(0)
			duration = (timestamp - start).total_seconds()

			durations[key].append(duration)
			matches[key].append((start, timestamp, duration))

for (user, node), pairs in matches.items():
    for start, end, duration in pairs[:5]:
        print(start.isoformat(), "->", end.isoformat(), duration)

print("Average ThoughtBox duration per user/node:")
for (user, node), values in durations.items():
	avg = sum(values) / len(values)
	print(f"{user:10} | {node:40} | count={len(values)}) | avg={avg:.4f} s")

anomalous_intervals = []

print("\nLong durations (>60 s):")
for (user, node), pairs in matches.items():
	for start, end, duration in pairs:
		if duration > 120:
			anomalous_intervals.append((user, node, start, end))

			print(
				f"User : {user}\n"
				f"Node : {node}\n"
				f"Start: {start.isoformat()}\n"
				f"End  : {end.isoformat()}\n"
				f"Δt   : {duration:.4f} s\n"
			)

2026-07-23T22:29:10.882000+00:00 -> 2026-07-23T22:29:10.884000+00:00 0.002
2026-07-23T22:29:10.891000+00:00 -> 2026-07-23T22:29:10.893000+00:00 0.002
2026-07-23T22:31:47.465000+00:00 -> 2026-07-23T22:31:47.467000+00:00 0.002
2026-07-23T22:31:47.507000+00:00 -> 2026-07-23T22:31:47.508000+00:00 0.001
2026-07-23T22:34:54.627000+00:00 -> 2026-07-23T22:34:54.628000+00:00 0.001
2026-07-23T22:34:54.631000+00:00 -> 2026-07-23T22:34:54.632000+00:00 0.001
2026-07-23T22:34:54.635000+00:00 -> 2026-07-23T22:34:54.636000+00:00 0.001
2026-07-23T22:37:15.426000+00:00 -> 2026-07-23T22:37:15.428000+00:00 0.002
2026-07-23T22:39:34.050000+00:00 -> 2026-07-23T22:39:34.052000+00:00 0.002
2026-07-23T22:39:34.055000+00:00 -> 2026-07-23T22:39:34.057000+00:00 0.002
2026-07-23T22:39:34.059000+00:00 -> 2026-07-23T22:39:34.060000+00:00 0.001
2026-07-23T22:41:55.834000+00:00 -> 2026-07-23T22:41:55.835000+00:00 0.001
2026-07-23T22:41:55.862000+00:00 -> 2026-07-23T22:41:55.863000+00:00 0.001
2026-07-23T22:41:55.86900

In [115]:
filtered_records = []

for record in records:
	user = record.get("actor", {}).get("account", {}).get("name")
	node = record.get("result", {}).get("extensions", {}).get(NODE_EXTENSION)

	timestamp = datetime.fromisoformat(record["timestamp"].replace("Z", "+00:00"))

	remove = False

	for bad_user, bad_node, start, end in anomalous_intervals:
		if user == bad_user and node == bad_node and start <= timestamp <= end:
			remove = True
			break

	if not remove:
		filtered_records.append(record)

removed_count = len(records) - len(filtered_records)

records = filtered_records

print(f"Removed records: {removed_count}")
print(f"Records after removing anomalous intervals: {len(records)}")


Removed records: 8
Records after removing anomalous intervals: 232


In [116]:
with open(gameplay_path, "w", encoding="utf-8") as f:
	for record in records:
		f.write(json.dumps(record, ensure_ascii=False))
		f.write("\n")
